# 03 — Backfill Historical Data

**Mục tiêu:** Cào dữ liệu quá khứ (50 ngày) từ Open-Meteo Archive API → Insert vào TimescaleDB.

**Tại sao cần backfill?**
- LSTM cần ≥ 48 giờ history để forecast
- Isolation Forest cần data quá khứ để học pattern bình thường
- Prophet cần ≥ 30 ngày để detect seasonality

**Số requests ước tính:**
- 63 provinces ÷ 1 request/province = **63 requests** (Archive API, mỗi request = 50 ngày)
- **RẤT ÍT** — nhưng Archive API có thể bị giới hạn trên free tier

**Quy trình:**
1. Test Archive API với 1 tỉnh (HCM)
2. Check data coverage hiện có
3. Backfill 50 ngày cho 63 tỉnh
4. Insert vào TimescaleDB
5. Verify data coverage

## Cell 1: Cài đặt

In [ ]:
# Install dependencies nếu cần
# !pip install asyncpg aiohttp pandas -q

import requests
import pandas as pd
import asyncpg
import asyncio
import aiohttp
import json
import time
import pprint
from datetime import datetime, timedelta
from typing import Dict, List, Optional
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# ── Cấu hình ──────────────────────────────────────────────────────────
ARCHIVE_URL  = 'https://archive-api.open-meteo.com/v1/archive'
WEATHER_URL = 'https://api.open-meteo.com/v1/forecast'
AQ_URL      = 'https://air-quality-api.open-meteo.com/v1/air-quality'

WEATHER_VARS = 'temperature_2m,wind_speed_10m,relative_humidity_2m,precipitation'
AQ_VARS      = 'pm10,pm2_5,nitrogen_dioxide,ozone,uv_index,us_aqi'

# TimescaleDB
DB_HOST     = 'localhost'
DB_PORT     = 5432
DB_USER     = 'reis'
DB_PASSWORD = 'reis_secret'
DB_NAME     = 'reis_db'

# Backfill config
BACKFILL_DAYS = 50  # Số ngày cần backfill

## Cell 2: Check data coverage hiện có

In [ ]:
async def check_current_coverage():
    """Kiểm tra data coverage hiện có trong TimescaleDB."""
    conn = await asyncpg.connect(
        host=DB_HOST, port=DB_PORT,
        user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
    )
    
    rows = await conn.fetch("""
        SELECT 
            province_id,
            COUNT(*)                   AS readings,
            MIN(time)                 AS first_reading,
            MAX(time)                 AS last_reading,
            COUNT(DISTINCT DATE(time)) AS days_covered
        FROM env_readings
        GROUP BY province_id
        ORDER BY province_id
    """)
    await conn.close()
    
    df = pd.DataFrame([dict(r) for r in rows])
    return df

# Chạy check
try:
    df_coverage = await check_current_coverage()
    print(f"Tổng số provinces có data: {len(df_coverage)}")
    display(df_coverage)
    
    if len(df_coverage) > 0:
        print(f"\nTổng readings: {df_coverage['readings'].sum()}")
        print(f"Min days covered: {df_coverage['days_covered'].min()}")
        print(f"Max days covered: {df_coverage['days_covered'].max()}")
        print(f"Mean days covered: {df_coverage['days_covered'].mean():.1f}")
except Exception as e:
    print(f"Lỗi kết nối DB: {e}")
    print("→ TimescaleDB có thể chưa chạy. Chạy: docker-compose up -d")

## Cell 3: Test Archive API với HCM (1 tỉnh)

In [ ]:
# Test Archive API cho HCM (province_id=2)
# Endpoint: https://archive-api.open-meteo.com/v1/archive

hcm_lat, hcm_lon = 10.7769, 106.7009

# Tính date range: 50 ngày trước
end_date   = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=BACKFILL_DAYS)).strftime('%Y-%m-%d')

print(f"Testing Archive API for HCM")
print(f"Date range: {start_date} → {end_date}")
print(f"Coords: ({hcm_lat}, {hcm_lon})")

# Test Weather Archive
params_weather = {
    'latitude':  hcm_lat,
    'longitude': hcm_lon,
    'start_date': start_date,
    'end_date':   end_date,
    'hourly':     WEATHER_VARS,
    'timezone':   'Asia/Ho_Chi_Minh',
}

r_weather = requests.get(ARCHIVE_URL, params=params_weather, timeout=60)
print(f"\nWeather Archive Status: {r_weather.status_code}")

if r_weather.status_code == 200:
    data_w = r_weather.json()
    print(f"Keys: {list(data_w.keys())}")
    hourly_w = data_w.get('hourly', {})
    print(f"Số giờ data (weather): {len(hourly_w.get('time', []))}")
    print(f"Time range: {hourly_w['time'][0]} → {hourly_w['time'][-1]}")
else:
    print(f"ERROR: {r_weather.text[:200]}")

In [ ]:
# Test Air Quality Archive
params_aq = {
    'latitude':  hcm_lat,
    'longitude': hcm_lon,
    'start_date': start_date,
    'end_date':   end_date,
    'hourly':     AQ_VARS,
    'timezone':   'Asia/Ho_Chi_Minh',
}

r_aq = requests.get(AQ_URL, params=params_aq, timeout=60)
print(f"AQ Archive Status: {r_aq.status_code}")

if r_aq.status_code == 200:
    data_aq = r_aq.json()
    hourly_aq = data_aq.get('hourly', {})
    print(f"Số giờ data (AQ): {len(hourly_aq.get('time', []))}")
    
    # Kiểm tra null values
    df_aq_sample = pd.DataFrame(hourly_aq)
    print(f"\nNull counts (HCM AQ sample):")
    display(df_aq_sample.isnull().sum())
else:
    print(f"ERROR: {r_aq.text[:200]}")

## Cell 4: Dữ liệu 63 tỉnh

In [ ]:
# Dữ liệu 63 tỉnh (từ config/constants.py)
PROVINCES = [
    (1,  'Hà Nội',         21.0285, 105.8542),
    (2,  'Hồ Chí Minh',    10.7769, 106.7009),
    (3,  'Hải Phòng',      20.8623, 106.6799),
    (4,  'Đà Nẵng',        16.0544, 108.2022),
    (5,  'Hà Giang',       22.8279, 104.9823),
    (6,  'Cao Bằng',       22.6761, 106.2016),
    (7,  'Lai Châu',       22.3862, 103.4702),
    (8,  'Lào Cai',        22.4962, 103.9680),
    (9,  'Tuyên Quang',    21.8212, 105.1833),
    (10, 'Lạng Sơn',       22.1398, 105.8320),
    (11, 'Bắc Kạn',        22.1398, 105.8320),
    (12, 'Thái Nguyên',    21.5954, 105.8387),
    (13, 'Yên Bái',        21.7049, 104.8791),
    (14, 'Sơn La',         21.3270, 103.9144),
    (15, 'Phú Thọ',        21.3135, 105.3946),
    (16, 'Vĩnh Phúc',      21.3079, 105.5965),
    (17, 'Quảng Ninh',     20.9489, 107.1035),
    (18, 'Bắc Giang',      21.2804, 106.1985),
    (19, 'Bắc Ninh',       21.2816, 106.1989),
    (20, 'Hải Dương',      20.9411, 106.3330),
    (21, 'Hưng Yên',       20.6626, 106.0585),
    (22, 'Hòa Bình',       20.8199, 105.3438),
    (23, 'Hà Nam',          20.5514, 105.9171),
    (24, 'Nam Định',        20.4272, 106.1749),
    (25, 'Thái Bình',       20.4480, 106.3435),
    (26, 'Ninh Bình',       20.2573, 105.9719),
    (27, 'Thanh Hóa',       19.7996, 105.7864),
    (28, 'Nghệ An',         18.6596, 105.6970),
    (29, 'Hà Tĩnh',         18.3393, 105.9029),
    (30, 'Quảng Bình',      19.6868, 105.7875),
    (31, 'Quảng Trị',       16.7468, 107.1877),
    (32, 'Thừa Thiên Huế',  16.4639, 107.5863),
    (33, 'Quảng Nam',       15.5752, 108.4743),
    (34, 'Quảng Ngãi',      14.3512, 108.0027),
    (35, 'Kon Tum',         13.8865, 109.1133),
    (36, 'Gia Lai',         13.7700, 109.2318),
    (37, 'Bình Định',       13.0467, 109.3108),
    (38, 'Phú Yên',         13.0467, 109.3108),
    (39, 'Đắk Lắk',         12.6797, 108.0447),
    (40, 'Đắk Nông',         12.0006, 107.6960),
    (41, 'Lâm Đồng',         11.9402, 108.4376),
    (42, 'Bình Phước',       11.5314, 106.8943),
    (43, 'Tây Ninh',         10.9460, 106.1900),
    (44, 'Bình Dương',       11.2943, 106.6750),
    (45, 'Đồng Nai',         10.9508, 106.8221),
    (46, 'Bình Thuận',       10.9378, 108.0912),
    (47, 'Khánh Hòa',        11.2349, 109.1941),
    (48, 'Ninh Thuận',       11.5770, 108.9865),
    (49, 'Long An',           10.5389, 106.4061),
    (50, 'Đồng Tháp',        10.3585, 106.3643),
    (51, 'An Giang',          10.3904, 105.4344),
    (52, 'Bà Rịa - Vũng Tàu', 10.4963, 107.1688),
    (53, 'Tiền Giang',       10.3606, 106.3658),
    (54, 'Kiên Giang',        9.9356,  106.3416),
    (55, 'Cần Thơ',           10.0362, 105.7873),
    (56, 'Hậu Giang',         9.7832,  105.4670),
    (57, 'Vĩnh Long',         9.9356,  106.3416),
    (58, 'Bến Tre',           10.2315, 106.3599),
    (59, 'Trà Vinh',          9.9356,  106.3416),
    (60, 'Sóc Trăng',         9.6025,  105.9731),
    (61, 'Bạc Liêu',         9.2869,  105.7228),
    (62, 'Cà Mau',            9.1762,  105.1508),
    (63, 'Điện Biên',        21.3924, 103.0160),
]

print(f"Tổng số tỉnh: {len(PROVINCES)}")
print(f"Backfill: {BACKFILL_DAYS} ngày")
print(f"Số requests ước tính: {len(PROVINCES)} × 2 (weather + AQ) = {len(PROVINCES)*2} requests")

## Cell 5: Backfill cho 1 tỉnh (Hà Nội) — Test trước khi chạy tất cả

In [ ]:
async def backfill_province(
    province_id: int,
    lat: float,
    lon: float,
    start_date: str,
    end_date: str,
    session: aiohttp.ClientSession,
) -> pd.DataFrame:
    """
    Backfill dữ liệu hourly cho 1 tỉnh trong date range.
    
    Gọi song song 2 Archive API (weather + AQ) cho 1 tỉnh.
    Merge theo index (cùng timestamp).
    
    Returns: DataFrame với 1 row mỗi giờ.
    """
    # Weather Archive
    weather_task = session.get(
        ARCHIVE_URL,
        params={
            'latitude': lat, 'longitude': lon,
            'start_date': start_date, 'end_date': end_date,
            'hourly': WEATHER_VARS,
            'timezone': 'Asia/Ho_Chi_Minh',
        },
        timeout=aiohttp.ClientTimeout(total=60),
    )
    
    # AQ Archive (dùng forecast endpoint vì AQ archive có thể không có)
    # Thử archive endpoint trước
    aq_task = session.get(
        AQ_URL,  # Hoặc ARCHIVE_URL nếu AQ archive khả dụng
        params={
            'latitude': lat, 'longitude': lon,
            'start_date': start_date, 'end_date': end_date,
            'hourly': AQ_VARS,
            'timezone': 'Asia/Ho_Chi_Minh',
        },
        timeout=aiohttp.ClientTimeout(total=60),
    )
    
    weather_resp, aq_resp = await asyncio.gather(weather_task, aq_task)
    
    weather_data = await weather_resp.json()
    aq_data     = await aq_resp.json()
    
    # Parse weather
    hw = weather_data.get('hourly', {})
    times_w = hw.get('time', [])
    
    # Parse AQ  
    ha = aq_data.get('hourly', {})
    times_aq = ha.get('time', [])
    
    if not times_w:
        logger.warning(f"Province {province_id}: No weather data returned")
        return pd.DataFrame()
    
    # Build DataFrames và merge trên time index
    df_w  = pd.DataFrame({'time': times_w})
    for key, vals in hw.items():
        if key != 'time':
            df_w[key] = vals
    
    df_aq = pd.DataFrame({'time': times_aq})
    for key, vals in ha.items():
        if key != 'time':
            df_aq[key] = vals
    
    # Merge trên time (inner join — chỉ giữ timestamp có cả 2)
    df = pd.merge(df_w, df_aq, on='time', how='inner')
    
    # Rename columns
    df = df.rename(columns={
        'temperature_2m':       'temperature',
        'relative_humidity_2m': 'humidity',
        'wind_speed_10m':       'wind_speed',
        'nitrogen_dioxide':     'no2',
        'us_aqi':              'aqi',
    })
    
    df['province_id'] = province_id
    
    return df

# ── Test với Hà Nội (province_id=1) ─────────────────────────────────────
hanoi = next(p for p in PROVINCES if p[0] == 1)
pid, pname, plat, plon = hanoi

print(f"Test backfill: {pname} (ID={pid})")

async with aiohttp.ClientSession() as session:
    df_test = await backfill_province(
        province_id=pid,
        lat=plat, lon=plon,
        start_date=start_date,
        end_date=end_date,
        session=session,
    )

print(f"Rows returned: {len(df_test)}")
print(f"Columns: {list(df_test.columns)}")
display(df_test.head(3))
display(df_test.tail(3))

# Null check
print(f"\nNull counts:")
display(df_test.isnull().sum())

## Cell 6: Insert vào TimescaleDB

In [ ]:
async def insert_hourly_data(df: pd.DataFrame) -> int:
    """
    Insert hourly data vào TimescaleDB.
    
    Returns: Số rows đã insert.
    """
    if df.empty:
        return 0
    
    pool = await asyncpg.create_pool(
        host=DB_HOST, port=DB_PORT,
        user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
        min_size=2, max_size=10,
    )
    
    INSERT_SQL = """
        INSERT INTO env_readings (
            time, province_id,
            temperature, humidity, wind_speed, precipitation,
            pm2_5, pm10, aqi, no2, ozone, uv_index,
            anomaly_score, is_anomaly, raw_json, inserted_at
        ) VALUES (
            $1, $2, $3, $4, $5, $6, $7, $8, $9, $10, $11, $12, $13, $14, $15, $16
        )
        ON CONFLICT DO NOTHING;
    """
    
    rows = []
    now = datetime.now()
    
    for _, row in df.iterrows():
        rows.append((
            row['time'],                   # $1
            int(row['province_id']),       # $2
            float(row['temperature']) if pd.notna(row.get('temperature')) else None,
            float(row['humidity'])    if pd.notna(row.get('humidity'))    else None,
            float(row['wind_speed'])   if pd.notna(row.get('wind_speed'))  else None,
            float(row['precipitation']) if pd.notna(row.get('precipitation')) else None,
            float(row['pm2_5'])       if pd.notna(row.get('pm2_5'))       else None,
            float(row['pm10'])         if pd.notna(row.get('pm10'))         else None,
            int(row['aqi'])            if pd.notna(row.get('aqi'))           else None,
            float(row['no2'])         if pd.notna(row.get('no2'))           else None,
            float(row['ozone'])       if pd.notna(row.get('ozone'))         else None,
            float(row['uv_index'])     if pd.notna(row.get('uv_index'))      else None,
            None,  # anomaly_score
            False, # is_anomaly
            None,  # raw_json
            now,   # inserted_at
        ))
    
    async with pool.acquire() as conn:
        await conn.executemany(INSERT_SQL, rows)
    
    await pool.close()
    return len(rows)

# ── Test insert với Hà Nội ────────────────────────────────────────────────
if len(df_test) > 0:
    inserted = await insert_hourly_data(df_test)
    print(f"Inserted {inserted} rows for Hà Nội")
    
    # Verify
    conn = await asyncpg.connect(
        host=DB_HOST, port=DB_PORT,
        user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
    )
    count = await conn.fetchval(
        "SELECT COUNT(*) FROM env_readings WHERE province_id = 1"
    )
    await conn.close()
    print(f"Total rows for Hà Nội in DB: {count}")

## Cell 7: BACKFILL TẤT CẢ 63 TỈNH

**⚠️ CHẠY CELL NÀY CẨN THẬN**

- Số requests: 63 provinces × 2 APIs = **126 requests**
- Thời gian ước tính: ~10-20 phút (tùy API rate limit)
- Nếu API rate limit → thêm sleep giữa các request

**Lưu ý:** Nếu Archive API bị limit, chỉ cần backfill **30 ngày** thay vì 50.

In [ ]:
async def backfill_all_provinces(
    provinces: List,
    start_date: str,
    end_date: str,
    sleep_between: float = 1.0,  # Giây giữa mỗi tỉnh (tránh rate limit)
) -> dict:
    """
    Backfill tất cả provinces.
    
    Returns: {
        'success': [province_ids],
        'failed':  [province_ids],
        'total_rows': int,
        'total_time_sec': float,
    }
    """
    results = {'success': [], 'failed': [], 'total_rows': 0}
    start_time = time.time()
    
    for i, (pid, pname, plat, plon) in enumerate(provinces, 1):
        print(f"\n[{i}/{len(provinces)}] {pname} (ID={pid})... ", end="", flush=True)
        
        try:
            async with aiohttp.ClientSession() as session:
                df = await backfill_province(
                    province_id=pid,
                    lat=plat, lon=plon,
                    start_date=start_date,
                    end_date=end_date,
                    session=session,
                )
            
            if df.empty:
                print("No data")
                results['failed'].append(pid)
                continue
            
            inserted = await insert_hourly_data(df)
            results['success'].append(pid)
            results['total_rows'] += inserted
            print(f"{len(df)} hourly rows, inserted={inserted}")
            
        except Exception as e:
            print(f"ERROR: {e}")
            results['failed'].append(pid)
        
        # Rate limit protection
        if i < len(provinces):
            await asyncio.sleep(sleep_between)
    
    results['total_time_sec'] = time.time() - start_time
    return results

# ── CHẠY BACKFILL ─────────────────────────────────────────────────────────
# Có thể giảm BACKFILL_DAYS nếu API bị limit
EFFECTIVE_DAYS = BACKFILL_DAYS  # Hoặc giảm xuống 30 nếu cần
effective_start = (datetime.now() - timedelta(days=EFFECTIVE_DAYS)).strftime('%Y-%m-%d')
effective_end   = datetime.now().strftime('%Y-%m-%d')

print(f"Backfilling {EFFECTIVE_DAYS} days: {effective_start} → {effective_end}")
print(f"Provinces: {len(PROVINCES)}")
print(f"Sleep between provinces: 1.0s")
print("=" * 60)

results = await backfill_all_provinces(
    provinces=PROVINCES,
    start_date=effective_start,
    end_date=effective_end,
    sleep_between=1.0,
)

print("\n" + "=" * 60)
print(f"BACKFILL COMPLETE!")
print(f"Success: {len(results['success'])}/{len(PROVINCES)} provinces")
print(f"Failed:  {len(results['failed'])}/{len(PROVINCES)} provinces")
print(f"Total rows inserted: {results['total_rows']:,}")
print(f"Time elapsed: {results['total_time_sec']/60:.1f} minutes")

if results['failed']:
    print(f"\nFailed province IDs: {results['failed']}")

## Cell 8: Verify Data Coverage sau Backfill

In [ ]:
# Kiểm tra lại data coverage
df_coverage_after = await check_current_coverage()
print(f"Tổng số provinces có data: {len(df_coverage_after)}")
display(df_coverage_after)

if len(df_coverage_after) > 0:
    print(f"\n📊 Coverage Summary:")
    print(f"  - Min days covered: {df_coverage_after['days_covered'].min()}")
    print(f"  - Max days covered: {df_coverage_after['days_covered'].max()}")
    print(f"  - Mean days covered: {df_coverage_after['days_covered'].mean():.1f}")
    print(f"  - Total readings: {df_coverage_after['readings'].sum():,}")
    
    # Provinces với < 30 ngày data
    low_coverage = df_coverage_after[df_coverage_after['days_covered'] < 30]
    if len(low_coverage) > 0:
        print(f"\n⚠️  Provinces với < 30 ngày data ({len(low_coverage)}):")
        display(low_coverage)
    else:
        print(f"\n✅ Tất cả provinces có ≥ 30 ngày data — sẵn sàng train ML!")

## Cell 9: AQI Distribution Check

In [ ]:
async def get_aqi_distribution():
    conn = await asyncpg.connect(
        host=DB_HOST, port=DB_PORT,
        user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
    )
    rows = await conn.fetch("""
        SELECT 
            province_id,
            AVG(aqi) as avg_aqi,
            MIN(aqi) as min_aqi,
            MAX(aqi) as max_aqi,
            STDDEV(aqi) as std_aqi
        FROM env_readings
        WHERE aqi IS NOT NULL
        GROUP BY province_id
        ORDER BY avg_aqi DESC
    """)
    await conn.close()
    return pd.DataFrame([dict(r) for r in rows])

df_aqi = await get_aqi_distribution()
print(f"AQI Statistics cho {len(df_aqi)} provinces:")
display(df_aqi.head(10))

print(f"\nTop 5 ô nhiễm nhất:")
display(df_aqi.head(5))

print(f"\n5 tỉnh sạch nhất:")
display(df_aqi.tail(5))